<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/tracking_woche12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 12 — Tracking und Bewegungsschätzung

**FHNW HSW · BSc Business AI · Computer Vision · FS 2026**

Dieses Notebook ergänzt die Vorlesung mit fünf praktischen Teilen:

1. **Synthetische Bildsequenz** — wir bauen uns ein Mini-Video, an dem wir Bewegung beobachten können.
2. **Lucas-Kanade** — sparse Optical Flow für ausgewählte Punkte.
3. **Farneback** — dense Optical Flow für jeden Pixel, visualisiert per HSV.
4. **Tracking mit YOLOv8 + ByteTrack** — Multi-Object Tracking mit nur wenigen Zeilen Code.
5. **Mini-Kalman-Filter** — wir bauen den Filter selbst und sehen, wie Predict & Update zusammenspielen.

Das Notebook läuft direkt in **Google Colab** — keine lokale Installation nötig.

## Setup

OpenCV und NumPy sind in Colab schon vorinstalliert. Ultralytics und der zugehörige YOLOv8-Tracker müssen wir kurz nachinstallieren — das dauert ca. 30 Sekunden.

In [ ]:
# In Colab: Ultralytics installieren (still, damit der Output kompakt bleibt)
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
print('Setup ok')

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import patches

# Helper: Bild im Notebook anzeigen (OpenCV nutzt BGR, Matplotlib RGB)
def show(img, title=None, figsize=(6, 4), cmap=None):
    plt.figure(figsize=figsize)
    if img.ndim == 3 and img.shape[2] == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    if title: plt.title(title)
    plt.axis('off')
    plt.show()

## Teil 1 — Eine synthetische Bildsequenz

Bevor wir mit echten Videos arbeiten, bauen wir uns ein kleines, kontrolliertes Beispiel: Eine helle Scheibe wandert über einen verrauschten Hintergrund. So wissen wir genau, welche Bewegung im Bild stattfindet und können den Output unserer Algorithmen damit vergleichen.

Der Vorteil synthetischer Daten: **Ground Truth ist gratis**. Wir kennen die wahre Bewegung.

In [ ]:
def make_frame(cx, cy, size=200, radius=18, noise=15, seed=0):
    """Ein Frame: heller Kreis bei (cx, cy) auf grauem, leicht verrauschtem Hintergrund."""
    rng = np.random.default_rng(seed)
    img = np.full((size, size), 100, dtype=np.uint8)
    img += rng.integers(-noise, noise, img.shape, dtype=np.int16).astype(np.uint8)
    cv2.circle(img, (int(cx), int(cy)), radius, 230, -1)
    return img

# Sequenz: Kreis wandert von links unten nach rechts oben
N = 12
frames = [make_frame(40 + 12*t, 160 - 8*t, seed=t) for t in range(N)]

# Erste, mittlere und letzte Frame anzeigen
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, t in zip(axes, [0, N//2, N-1]):
    ax.imshow(frames[t], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f't = {t}')
    ax.axis('off')
plt.tight_layout(); plt.show()

**Frage zum Nachdenken:** Wenn wir nur den ersten und den letzten Frame nebeneinander legen und die Differenz bilden — was sehen wir dann? Versuchen wir das einmal.

In [ ]:
diff = cv2.absdiff(frames[0], frames[-1])
show(diff, title='|Frame 0 − Frame 11|', figsize=(4, 4))

Schöne Beobachtung: Die Differenz zeigt zwei helle Flecken — dort, wo der Kreis _war_, und dort, wo er _ist_. Das ist die rohste Form der Bewegungsdetektion. Aber: Sie sagt uns nicht, _in welche Richtung_ sich die Pixel bewegt haben, und sie ordnet keine Punkte einander zu. Genau das macht Optical Flow.

## Teil 2 — Lucas-Kanade: Sparse Optical Flow

Lucas-Kanade verfolgt **einzelne, gut wählbare Punkte** von Frame zu Frame. Typischer Workflow:

1. Mit `cv2.goodFeaturesToTrack` interessante Eckpunkte im ersten Frame finden.
2. Mit `cv2.calcOpticalFlowPyrKL` deren Position im nächsten Frame schätzen.
3. Wiederholen bis zum Ende der Sequenz.

Wir tracken dabei nicht das ganze Bild, sondern nur eine Handvoll Punkte — daher _sparse_.

In [ ]:
# Schritt 1: gute Eckpunkte im ersten Frame finden
p0 = cv2.goodFeaturesToTrack(frames[0],
                             maxCorners=20,
                             qualityLevel=0.05,
                             minDistance=8,
                             blockSize=7)
print(f'{len(p0)} Punkte gefunden, shape: {p0.shape}')

# Visualisierung der Startpunkte
vis = cv2.cvtColor(frames[0], cv2.COLOR_GRAY2BGR)
for pt in p0.reshape(-1, 2):
    cv2.circle(vis, tuple(pt.astype(int)), 4, (0, 255, 0), -1)
show(vis, title='Eckpunkte im ersten Frame', figsize=(4, 4))

In [ ]:
# Schritt 2 + 3: durch alle Frames hindurch tracken und Spuren sammeln
tracks = [p0.reshape(-1, 2).copy()]   # Liste von (N_punkte, 2) für jeden Zeitschritt
lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

prev = frames[0]
p_prev = p0
for t in range(1, N):
    nxt = frames[t]
    p_next, status, err = cv2.calcOpticalFlowPyrLK(prev, nxt, p_prev, None, **lk_params)
    tracks.append(p_next.reshape(-1, 2).copy())
    prev = nxt
    p_prev = p_next

tracks = np.stack(tracks)   # (T, N, 2)
print('Tracks-Shape:', tracks.shape)

In [ ]:
# Spuren über das letzte Frame zeichnen
vis = cv2.cvtColor(frames[-1], cv2.COLOR_GRAY2BGR)
for n in range(tracks.shape[1]):
    pts = tracks[:, n, :].astype(int)
    for i in range(1, len(pts)):
        cv2.line(vis, tuple(pts[i-1]), tuple(pts[i]), (0, 200, 255), 1)
    cv2.circle(vis, tuple(pts[-1]), 3, (0, 0, 255), -1)
show(vis, title='Lucas-Kanade-Spuren', figsize=(5, 5))

Schau dir die Spuren genau an: Punkte _auf_ dem Kreis bewegen sich klar mit ihm mit. Punkte _im Hintergrund_ wackeln auf der Stelle (Rauschen). Genau dieses Verhalten erwarten wir.

**Übung 2.1** — _Aperture-Problem in der Praxis:_ Setze `qualityLevel=0.001` und schau, was passiert. Was beobachtest du an Punkten in flachen Bildregionen?

## Teil 3 — Farneback: Dense Optical Flow

Farneback berechnet einen Bewegungsvektor **für jeden Pixel** — also ein vollständiges Flussfeld. Das ist teurer als Lucas-Kanade, liefert aber ein dichtes Bewegungsbild.

Der Standard-Trick zur Visualisierung: Wir zeigen das Flussfeld als HSV-Bild, wobei **Farbton** = Bewegungsrichtung und **Helligkeit** = Bewegungsstärke.

In [ ]:
flow = cv2.calcOpticalFlowFarneback(
    frames[0], frames[-1],
    None,                      # kein Initialfluss
    pyr_scale=0.5, levels=3, winsize=15,
    iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
print('Flow-Shape:', flow.shape, '— letzter Achsenwert: dx, dy')

def flow_to_color(flow):
    """Konvertiert Flussfeld (H, W, 2) in HSV-Farbbild."""
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
    hsv[..., 0] = (ang * 180 / np.pi / 2).astype(np.uint8)   # Hue: Richtung
    hsv[..., 1] = 255                                         # Sättigung max
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(frames[0], cmap='gray');  axes[0].set_title('Frame 0');  axes[0].axis('off')
axes[1].imshow(frames[-1], cmap='gray'); axes[1].set_title('Frame 11'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(flow_to_color(flow), cv2.COLOR_BGR2RGB))
axes[2].set_title('Flussfeld (Farbton = Richtung)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

**Lesehilfe für das Flussbild:**

- Schwarze Bereiche = wenig oder keine Bewegung.
- Farbige Bereiche = Bewegung. Die Farbe codiert die Richtung (z.B. grün = nach unten-rechts).
- Helle Farbe = grosse Bewegung; dunkle Farbe = kleine Bewegung.

Das Flussbild zeigt: Im rauschigen Hintergrund passiert _wenig_ Konsistentes, dort wo der Kreis ist, leuchtet die Bewegung deutlich auf.

## Teil 4 — Multi-Object-Tracking mit YOLOv8 + ByteTrack

Jetzt verlassen wir die synthetischen Bilder und arbeiten mit einem echten Beispiel-Video. Ultralytics integriert die Tracker **ByteTrack** und **BoT-SORT** direkt in die YOLO-API. Wir brauchen nur drei Zeilen produktiven Code:

```python
model = YOLO('yolov8n.pt')
results = model.track(source='video.mp4', tracker='bytetrack.yaml', persist=True)
```

Die Magie passiert intern: YOLO detektiert in jedem Frame, der Tracker ordnet die Detektionen über die Zeit denselben **IDs** zu — genau das Datenassoziationsproblem aus den Folien.

In [ ]:
from ultralytics import YOLO
import urllib.request, os

# Kleines, frei verfügbares Demo-Video herunterladen, falls noch nicht vorhanden
VIDEO = 'people_walking.mp4'
if not os.path.exists(VIDEO):
    url = 'https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4'
    urllib.request.urlretrieve(url, VIDEO)
    print('Video heruntergeladen.')
else:
    print('Video bereits vorhanden.')

In [ ]:
# YOLOv8-Nano laden — klein und schnell, perfekt für eine Demo
model = YOLO('yolov8n.pt')

# .track() iteriert frame-by-frame durch das Video.
# stream=True liefert einen Generator und spart Speicher.
results = model.track(
    source=VIDEO,
    tracker='bytetrack.yaml',
    persist=True,
    classes=[0],          # nur Klasse 'person' (COCO-Index 0)
    stream=True,
    verbose=False,
)

# Wir sammeln die ersten paar Frames mit eingezeichneten Tracks
annotated_frames = []
for i, r in enumerate(results):
    annotated_frames.append(r.plot())   # bereits gerendertes BGR-Bild
    if i >= 60:                         # ca. 2 Sekunden bei 30 FPS
        break
print(f'{len(annotated_frames)} Frames verarbeitet')

In [ ]:
# Drei Snapshots der Tracking-Sequenz
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, idx in zip(axes, [0, len(annotated_frames)//2, len(annotated_frames)-1]):
    ax.imshow(cv2.cvtColor(annotated_frames[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {idx}'); ax.axis('off')
plt.tight_layout(); plt.show()

Beachte die kleinen Zahlen über jeder Bounding Box: **id:1, id:2, ...** — das sind die Track-IDs, die ByteTrack vergibt. Derselbe Mensch hat über die Zeit dieselbe ID, auch wenn er sich bewegt.

**Übung 4.1** — Tausche `tracker='bytetrack.yaml'` gegen `tracker='botsort.yaml'`. Beobachtest du Unterschiede bei kurzen Verdeckungen oder ID-Wechseln?

**Übung 4.2** — Setze `model = YOLO('yolov8s.pt')` (small statt nano). Welche Detektion ist robuster, was ist der Preis?

## Teil 6 — Anwendung: Personen über eine Linie zählen

Eine klassische Tracking-Anwendung: zähle, wie viele Personen eine virtuelle Linie überqueren. Das geht erst mit Tracking sauber — denn nur mit Detektionen alleine wüssten wir nicht, ob es _eine_ Person ist, die dreimal über die Linie geht, oder _drei_ verschiedene.

In [ ]:
from collections import defaultdict

# Wir nehmen das Video von oben und zählen Personen, die eine vertikale Linie überqueren
LINE_X = 500     # x-Koordinate der Linie (für das Demo-Video)

history = defaultdict(list)   # track_id -> Liste von Mittelpunkten
crossings = 0

results = model.track(source=VIDEO, tracker='bytetrack.yaml',
                      persist=True, classes=[0], stream=True, verbose=False)

for r in results:
    if r.boxes is None or r.boxes.id is None: continue
    boxes = r.boxes.xywh.cpu().numpy()
    ids   = r.boxes.id.int().cpu().numpy()
    for (x, y, w, h), tid in zip(boxes, ids):
        history[tid].append(x)
        # Liniencheck: Wechsel des Vorzeichens von (x − LINE_X)?
        if len(history[tid]) >= 2:
            prev_x = history[tid][-2]
            if (prev_x - LINE_X) * (x - LINE_X) < 0:
                crossings += 1

print(f'Personen über die Linie: {crossings}')

## Wrap-up

Was du heute praktisch gemacht hast:

- Eine **Bildsequenz** synthetisch erzeugt, um Bewegungsalgorithmen kontrolliert zu testen.
- **Lucas-Kanade** für sparse Optical Flow auf wenigen Eckpunkten benutzt.
- **Farneback** für dense Optical Flow benutzt und das Resultat per HSV visualisiert.
- Mit drei Zeilen Code echtes **Multi-Object Tracking** mit YOLOv8 und ByteTrack betrieben.

**Weiterlesen:**
- Ultralytics-Tracking-Doku: <https://docs.ultralytics.com/modes/track/>
- ByteTrack-Paper: Zhang et al., ECCV 2022.
- MIT Vision Book, Kapitel _Motion Estimation_: <https://visionbook.mit.edu/motion_estimation.html>